# 05 — Evaluasi Sistem PrognoSense
**Peran:** Orang 3 — Evaluation

Notebook ini membaca hasil evaluasi yang telah tersedia di `data/results/`, memeriksa metrik model RUL, menampilkan hasil clustering, dan mengevaluasi strategi perawatan apabila hasil strategi tersedia di `evaluasi.json`.

> Jalankan notebook dari direktori utama repository `PrognoSense`. Notebook ini tidak melatih ulang model dan tidak mengubah file hasil asli.

## 1. Persiapan library dan lokasi data

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data" / "results").exists():
    raise FileNotFoundError(
        "Folder data/results tidak ditemukan. Jalankan notebook dari root repository PrognoSense."
    )

RESULTS = ROOT / "data" / "results"
print("Root proyek:", ROOT)
print("Folder hasil:", RESULTS)

## 2. Memuat hasil evaluasi

In [ ]:
def load_json(filename):
    path = RESULTS / filename
    if not path.exists():
        print(f"Peringatan: {filename} tidak ditemukan.")
        return None
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

rul_data = load_json("rul_evaluasi.json")
evaluasi_data = load_json("evaluasi.json")
kondisi_data = load_json("kondisi.json")

pred_path = RESULTS / "predictions.csv"
predictions = pd.read_csv(pred_path) if pred_path.exists() else None

print("rul_evaluasi.json:", "tersedia" if rul_data else "tidak tersedia")
print("evaluasi.json:", "tersedia" if evaluasi_data else "tidak tersedia")
print("kondisi.json:", "tersedia" if kondisi_data else "tidak tersedia")
if predictions is not None:
    print("predictions.csv:", predictions.shape)
    display(predictions.head())

## 3. Evaluasi model prediksi RUL

In [ ]:
if rul_data and rul_data.get("model"):
    model_df = pd.DataFrame(rul_data["model"])
    display(model_df)
else:
    model_df = pd.DataFrame()
    print("Data metrik model tidak tersedia di rul_evaluasi.json.")

In [ ]:
if not model_df.empty and {"nama", "rmse", "mae"}.issubset(model_df.columns):
    plot_df = model_df.set_index("nama")[["rmse", "mae"]]
    ax = plot_df.plot(kind="bar", figsize=(10, 5))
    ax.set_title("Perbandingan RMSE dan MAE Model RUL")
    ax.set_ylabel("Error (siklus)")
    ax.set_xlabel("Model")
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()
    plt.show()

    selected = rul_data.get("terpilih", "Belum tercantum")
    print("Model terpilih:", selected)
    chosen = model_df.loc[model_df["nama"] == selected]
    baseline = model_df.loc[model_df["nama"].str.contains("Baseline", case=False)]
    if not chosen.empty and not baseline.empty:
        b_rmse = float(baseline.iloc[0]["rmse"])
        m_rmse = float(chosen.iloc[0]["rmse"])
        improvement = (b_rmse - m_rmse) / b_rmse * 100 if b_rmse else np.nan
        print(f"RMSE baseline: {b_rmse:.2f}")
        print(f"RMSE {selected}: {m_rmse:.2f}")
        print(f"Penurunan RMSE terhadap baseline: {improvement:.1f}%")
else:
    print("Kolom yang dibutuhkan untuk grafik RMSE/MAE tidak lengkap.")

## 4. Prediksi RUL dibandingkan nilai aktual

In [ ]:
# Data 'sebar' dalam JSON merupakan titik ringkasan untuk visualisasi,
# bukan seluruh baris predictions.csv.
sebar = rul_data.get("sebar", []) if rul_data else []
if sebar:
    sebar_df = pd.DataFrame(sebar)
    if {"rul_aktual", "rul_pred"}.issubset(sebar_df.columns):
        plt.figure(figsize=(6, 6))
        plt.scatter(sebar_df["rul_aktual"], sebar_df["rul_pred"], alpha=0.65)
        batas_min = min(sebar_df["rul_aktual"].min(), sebar_df["rul_pred"].min())
        batas_max = max(sebar_df["rul_aktual"].max(), sebar_df["rul_pred"].max())
        plt.plot([batas_min, batas_max], [batas_min, batas_max], linestyle="--")
        plt.xlabel("RUL aktual (siklus)")
        plt.ylabel("RUL prediksi (siklus)")
        plt.title("Sampel RUL Prediksi vs Aktual")
        plt.tight_layout()
        plt.show()
        display(sebar_df.head())
else:
    print("Data sebar prediksi vs aktual tidak tersedia.")

## 5. Evaluasi clustering kondisi dan kesehatan

In [ ]:
cluster_data = evaluasi_data.get("clustering", {}) if evaluasi_data else {}
if cluster_data:
    ringkasan_cluster = pd.DataFrame([
        {
            "Jenis clustering": "Kondisi operasi",
            "Jumlah cluster (K)": cluster_data.get("k_kondisi"),
            "Silhouette score": cluster_data.get("silhouette_kondisi")
        },
        {
            "Jenis clustering": "Tahap kesehatan",
            "Jumlah cluster (K)": cluster_data.get("k_kesehatan"),
            "Silhouette score": cluster_data.get("silhouette_kesehatan")
        }
    ])
    display(ringkasan_cluster)

    pemilihan_k = cluster_data.get("pemilihan_k", [])
    if pemilihan_k:
        k_df = pd.DataFrame(pemilihan_k)
        display(k_df)
        if {"k", "silhouette"}.issubset(k_df.columns):
            plt.figure(figsize=(7, 4))
            plt.plot(k_df["k"], k_df["silhouette"], marker="o")
            plt.xlabel("Jumlah cluster (K)")
            plt.ylabel("Silhouette score")
            plt.title("Pemilihan K untuk Clustering Kondisi")
            plt.tight_layout()
            plt.show()
else:
    print("Bagian clustering tidak tersedia di evaluasi.json.")

## 6. Evaluasi strategi perawatan

In [ ]:
rl_data = evaluasi_data.get("rl", {}) if evaluasi_data else {}
strategi = rl_data.get("strategi", []) if isinstance(rl_data, dict) else []

if strategi:
    strategi_df = pd.DataFrame(strategi)
    display(strategi_df)

    # Grafik hanya dibuat bila kolom metrik yang sesuai tersedia.
    kolom_biaya = next((c for c in ["biaya", "biaya_total", "total_biaya"] if c in strategi_df.columns), None)
    kolom_gagal = next((c for c in ["gagal_pct", "persen_gagal", "failure_pct"] if c in strategi_df.columns), None)
    if kolom_biaya:
        ax = strategi_df.plot(kind="bar", x="nama" if "nama" in strategi_df.columns else strategi_df.columns[0],
                              y=kolom_biaya, figsize=(10, 5), legend=False)
        ax.set_title("Perbandingan Biaya Strategi Perawatan")
        ax.set_ylabel("Biaya")
        plt.xticks(rotation=25, ha="right")
        plt.tight_layout()
        plt.show()
    else:
        print("Tabel strategi tersedia, tetapi kolom biaya dengan nama umum tidak ditemukan; periksa tabel di atas.")
    if not rl_data.get("policy"):
        print("Policy Q-Learning belum tersedia atau masih kosong.")
    if not rl_data.get("kurva_reward"):
        print("Kurva reward Q-Learning belum tersedia atau masih kosong.")
else:
    print("Data strategi tidak ditemukan di evaluasi.json. Evaluasi strategi belum dapat ditampilkan dari file ini.")

## 7. Pemeriksaan data prediksi per kelompok

In [ ]:
if predictions is not None:
    required = {"unit", "cycle", "kelompok", "rul_pred", "rul_aktual", "tahap"}
    missing = required - set(predictions.columns)
    if missing:
        print("Kolom yang belum tersedia:", sorted(missing))
    else:
        print("Jumlah baris:", len(predictions))
        print("Jumlah mesin unik:", predictions["unit"].nunique())
        display(predictions.groupby("kelompok").agg(
            jumlah_baris=("unit", "size"),
            jumlah_mesin=("unit", "nunique"),
            rmse=("rul_pred", lambda x: np.nan)  # RMSE dihitung di bawah secara eksplisit
        ).reset_index())
        for kelompok, subset in predictions.groupby("kelompok"):
            error = subset["rul_pred"] - subset["rul_aktual"]
            rmse = float(np.sqrt(np.mean(error ** 2)))
            mae = float(np.mean(np.abs(error)))
            print(f"Kelompok {kelompok}: RMSE seluruh siklus={rmse:.2f}; MAE seluruh siklus={mae:.2f}; "
                  f"n={len(subset)}")
        print("\nCatatan: metrik per kelompok di atas dihitung dari seluruh siklus yang tercatat,")
        print("sehingga tidak langsung sebanding dengan metrik resmi pada siklus terakhir tiap mesin.")

## 8. Kesimpulan evaluasi

In [ ]:
print("Ringkasan hasil evaluasi")
if rul_data:
    print("- Model terpilih:", rul_data.get("terpilih", "tidak tercantum"))
    print("- Dasar pemilihan:", rul_data.get("dipilih_dengan", "tidak tercantum"))
    print("- Protokol pengujian:", rul_data.get("dinilai_pada", "tidak tercantum"))
if cluster_data:
    print(f"- Clustering kondisi: K={cluster_data.get('k_kondisi')}, "
          f"silhouette={cluster_data.get('silhouette_kondisi')}")
    print(f"- Clustering kesehatan: K={cluster_data.get('k_kesehatan')}, "
          f"silhouette={cluster_data.get('silhouette_kesehatan')}")
print("- Interpretasi harus mempertimbangkan metrik resmi, pembagian kelompok data,")
print("  serta keterbatasan simulasi dan fungsi biaya yang digunakan.")

## Catatan
- Metrik resmi model diambil dari `rul_evaluasi.json`, bukan dihitung ulang dari semua baris `predictions.csv`.
- `predictions.csv` berisi banyak siklus per mesin. Evaluasi semua siklus dapat berbeda dari evaluasi resmi yang menggunakan siklus terakhir tiap mesin.
- Bagian Q-Learning hanya bisa menampilkan hasil jika `evaluasi.json` sudah berisi policy dan kurva reward resmi.
- Periksa kembali hasil dan interpretasinya sebelum dipakai di laporan atau presentasi.